# 🔬 Notebook 3: Distributed Lock Manager — Deep Dive: making the lock actually safe

## 🛠️ Setup

```bash
cd 06-system-designs/distributed-lock-manager
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we'll build

A toy `LockManager` plus a `FencedResource` (the "protected thing", e.g. a database row),
and then we reproduce **three** ways a plausible-looking lock loses mutual exclusion.
Each one is shown failing first, then fixed:

| # | Failure | Fix |
|---|---|---|
| 1 | **GC pause → split brain**: the lease expires while the holder is paused, and its stale write still lands | resource checks a **fencing token** |
| 2 | **`release` with no owner check**: anyone can delete anyone's lock | owner + token on release |
| 3 | **Non-atomic check-then-delete**: `GET` says "mine", the lease expires, `DEL` kills the *next* owner's lock | atomic **compare-and-delete** (Redis Lua / etcd txn / SQL `WHERE owner=?`) |

Then we look at lease renewal, and what it costs.

In [ ]:
# Toy lock manager (same shape as Notebook 2's InMemoryLockService, plus renew()).
import time, threading
from dataclasses import dataclass

@dataclass
class LockEntry:
    owner: str
    token: int
    expires_at: float

class LockManager:
    def __init__(self):
        self._locks: dict[str, LockEntry] = {}
        self._next_token = 0
        self._mu = threading.Lock()

    def _now(self): return time.time()

    def acquire(self, name, owner, ttl_s):
        with self._mu:
            e = self._locks.get(name)
            if e and e.expires_at > self._now() and e.owner != owner:
                return None  # held by someone else
            self._next_token += 1
            e = LockEntry(owner, self._next_token, self._now() + ttl_s)
            self._locks[name] = e
            return {"token": e.token, "expires_at": e.expires_at}

    def renew(self, name, owner, token, ttl_s):
        with self._mu:
            e = self._locks.get(name)
            if not e or e.owner != owner or e.token != token: return None
            if e.expires_at < self._now(): return None
            e.expires_at = self._now() + ttl_s
            return {"expires_at": e.expires_at}

    def release(self, name, owner, token):
        with self._mu:
            e = self._locks.get(name)
            if e and e.owner == owner and e.token == token:
                del self._locks[name]; return True
            return False

lm = LockManager()
a1 = lm.acquire("nightly-job", "worker-A", ttl_s=1.0)
print("A acquires:", a1)
print("B denied  :", lm.acquire("nightly-job", "worker-B", ttl_s=1.0))
time.sleep(1.1)  # let A's lease expire
a2 = lm.acquire("nightly-job", "worker-B", ttl_s=1.0)
print("B after expiry:", a2, "← note the token is strictly greater")


## 🚫 Bad: TTL-only lock, resource trusts the caller

Simulate a GC pause: worker A acquires, then "pauses" (sleeps longer than the TTL).
Worker B notices A's lease expired and takes over. A wakes up **still believing it
holds the lock** and writes to the shared resource. Without a fencing check, that
write is accepted and **corrupts the data**.


In [ ]:
class UncheckedResource:
    """A resource that trusts anyone who calls it (the 'bad' design)."""
    def __init__(self): self.state = []
    def write(self, who, data):
        self.state.append((who, data))
        print(f"  ACCEPT (unchecked): {who} wrote {data!r}")

lm = LockManager()
res = UncheckedResource()

# A acquires with a short 0.5s TTL, then goes into a long "GC pause"
a = lm.acquire("job", "A", ttl_s=0.5)
print("A acquired token=", a["token"])
print("A starts pause (1.2s) — imagine a stop-the-world GC")
time.sleep(1.2)

# While A was paused, B's heartbeat noticed expiry and B acquired
b = lm.acquire("job", "B", ttl_s=5.0)
print("B acquired token=", b["token"])
res.write("B", "correct-result")

# A wakes up, still believes it holds the lock, and writes
print("A wakes up and writes (DANGER):")
res.write("A", "stale-result-from-before-pause")

print("final state →", res.state)
print("❌ corrupted: B's correct write is followed by A's stale write")


## 🏆 Best: resource checks the fencing token

Now the resource remembers the **highest token** it has ever seen and rejects any
write carrying a lower token. A's stale write from before the pause can't win
because its token is smaller than B's.


In [ ]:
class FencedResource:
    """A resource that only accepts writes with a monotonically non-decreasing token."""
    def __init__(self):
        self.last_token = 0
        self.state = []
    def write(self, token, who, data):
        if token < self.last_token:
            print(f"  REJECT: token {token} from {who} < last_seen {self.last_token}")
            return False
        self.last_token = token
        self.state.append((token, who, data))
        print(f"  ACCEPT: token {token} from {who} wrote {data!r}")
        return True

lm = LockManager()
res = FencedResource()

a = lm.acquire("job", "A", ttl_s=0.5)
print("A acquired token=", a["token"])
time.sleep(1.2)  # GC pause

b = lm.acquire("job", "B", ttl_s=5.0)
print("B acquired token=", b["token"])
res.write(b["token"], "B", "correct-result")

# A wakes up and tries to use its OLD token → rejected
print("A wakes up and tries to write with its stale token:")
res.write(a["token"], "A", "stale-result-from-before-pause")

print("final state →", res.state)
print("✅ safe: A's stale write was rejected because a_token < b_token")


## 🚫 Bad: `release` that doesn't check the owner

Fencing protects the *resource*. It does nothing for the **lock service itself** if
`release` is sloppy. Two bugs show up in almost every hand-rolled implementation:

1. **No owner check** — anybody can delete anybody's lock key.
2. **Non-atomic check-then-delete** — you `GET` the key, see your own id, and `DEL` it.
   Between the GET and the DEL your lease can expire and someone else can acquire.
   You then delete *their* lock. This is the bug the Redis docs' Lua script exists to
   prevent, and it is invisible in single-threaded testing.

Let's make both fail on purpose.

In [ ]:
# 🚫 A lock service whose release() is a blind DELETE -- no owner, no token.
class UnsafeReleaseLockManager(LockManager):
    def release_blind(self, name):
        with self._mu:
            return self._locks.pop(name, None) is not None

lm = UnsafeReleaseLockManager()
a = lm.acquire("job", "A", ttl_s=30)
print("A holds the lock, token =", a["token"])

# B never held it -- B just calls release. A buggy retry loop does exactly this.
print("B calls release_blind():", lm.release_blind("job"))
c = lm.acquire("job", "C", ttl_s=30)
print("C now acquires        :", c is not None, "<- A is still working, thinking it holds the lock")
assert c is not None, "the unsafe version must let a stranger steal the lock"
print("\n❌ A and C both believe they hold 'job'. Mutual exclusion is gone.")

# ✅ The owner+token check refuses the same call.
lm2 = LockManager()
a2 = lm2.acquire("job", "A", ttl_s=30)
print("\nchecked release, wrong owner:", lm2.release("job", "B", a2["token"]))
print("checked release, right owner:", lm2.release("job", "A", a2["token"]))
print("✅ release is owner-scoped, so it is also idempotent: a second call just returns False.")
print("   second call                 :", lm2.release("job", "A", a2["token"]))

### The subtler one: check-then-delete is not atomic

Here the client *does* check ownership — it just does it in two steps:

```python
if redis.get("lock:job") == my_owner_id:   # step 1
    redis.delete("lock:job")               # step 2  <-- lease may have expired in between
```

We reproduce it by pausing between the two steps for longer than the TTL. Real life
supplies that pause for free: a GC pause, a slow syscall, or a retried network round-trip.

In [ ]:
# Reproduce the check-then-delete race, then fix it with an atomic compare-and-delete.
import threading, time

class KV:
    """A single-threaded key/value store, like one Redis shard. Each method is atomic;
    a *sequence* of methods is not -- which is the whole point of this cell."""
    def __init__(self):
        self.d: dict[str, tuple[str, float]] = {}   # key -> (owner, expires_at)
        self.mu = threading.Lock()

    def set_nx_px(self, key, owner, ttl_s):         # Redis: SET key owner NX PX ttl
        with self.mu:
            cur = self.d.get(key)
            if cur and cur[1] > time.time():
                return False
            self.d[key] = (owner, time.time() + ttl_s)
            return True

    def get(self, key):                             # Redis: GET key
        with self.mu:
            cur = self.d.get(key)
            if cur and cur[1] <= time.time():
                del self.d[key]; return None
            return cur[0] if cur else None

    def delete(self, key):                          # Redis: DEL key
        with self.mu:
            return self.d.pop(key, None) is not None

    def compare_and_delete(self, key, owner):       # Redis: the EVAL Lua script
        with self.mu:                               # GET + DEL inside ONE atomic step
            cur = self.d.get(key)
            if cur and cur[0] == owner and cur[1] > time.time():
                del self.d[key]; return True
            return False


def run(release_style: str) -> str:
    kv = KV()
    kv.set_nx_px("lock:job", "A", ttl_s=0.2)        # A takes a short lease

    # A stalls past its own TTL between the check and the delete.
    owner_seen = kv.get("lock:job")                 # step 1: "yes, it's mine"
    time.sleep(0.3)                                 # <-- lease expires here

    # Meanwhile B legitimately acquires the now-free lock.
    assert kv.set_nx_px("lock:job", "B", ttl_s=5), "B should get the expired lock"

    # A wakes up and finishes its release.
    if release_style == "check_then_delete":
        if owner_seen == "A":
            kv.delete("lock:job")                   # step 2: deletes B's lock!
    else:
        kv.compare_and_delete("lock:job", "A")      # atomic: sees owner is B, refuses

    return kv.get("lock:job")                       # who holds it now?


print("🚫 check-then-delete -> holder after A's release:", run("check_then_delete"))
print("✅ atomic CAS-delete -> holder after A's release:", run("atomic"))

assert run("check_then_delete") is None, "the unsafe version must destroy B's lock"
assert run("atomic") == "B", "the atomic version must leave B holding the lock"
print("\n❌ Unsafe: A deleted a lock it no longer owned -- B now runs unprotected,")
print("   and a third client can acquire on top of B.")
print("✅ Safe: compare-and-delete is one atomic step, so A's stale release is a no-op.")
print("\nIn Redis this is the EVAL script; in etcd it is a transaction guarded on the")
print("lease id; in SQL it is  DELETE ... WHERE name=? AND owner=? AND expires_at>now().")

## Lease renewal — how long-running jobs stay safe

Picking a TTL is a tradeoff:
- **Too long** → if a holder crashes, everyone waits forever.
- **Too short** → a slow-but-healthy job loses the lock mid-work.

The standard fix is a **heartbeat**: pick a short TTL (say 10s) and renew every
few seconds. If the holder dies, the lease expires quickly; if it's alive, it
keeps extending.


In [ ]:
# Heartbeat demo: A holds a 0.5s lease and renews every 0.1s for ~1 second.
# B, which just polls acquire(), should NEVER get in.
# Renewing at 1/5 of the TTL is the standard rule of thumb -- it tolerates several
# lost heartbeats (and a slow scheduler) before the lease actually lapses.
import threading, time

lm = LockManager()
stop = threading.Event()
lost_lease = threading.Event()

TTL, RENEW_EVERY = 0.5, 0.1

def heartbeat(name, owner, token):
    while not stop.is_set():
        if lm.renew(name, owner, token, ttl_s=TTL) is None:
            lost_lease.set()
            print("  heartbeat: lost the lease!")
            return
        time.sleep(RENEW_EVERY)

a = lm.acquire("job", "A", ttl_s=TTL)
hb = threading.Thread(target=heartbeat, args=("job", "A", a["token"]))
hb.start()

b_wins = 0
for i in range(10):
    time.sleep(0.1)
    if lm.acquire("job", "B", ttl_s=TTL):
        b_wins += 1
    print(f"t={(i+1)*0.1:.1f}s  B got lock? {b_wins > 0}")

stop.set(); hb.join()
assert b_wins == 0 and not lost_lease.is_set(), "a renewing holder must not lose the lock"
print("A finishes and releases:", lm.release("job", "A", a["token"]))
print("\nTrade-off: renewing costs a write every 0.1s *per held lock*, forever.")
print("That heartbeat traffic -- not acquires -- is what sizes the cluster (see NB1).")

## Redis `SET NX PX` — the pragmatic minimum

If you don't need consensus-level guarantees, Redis is the common choice.

```python
# Acquire (atomic on a single Redis):
SET lock:nightly-job worker-A NX PX 30000

# Release must check ownership — use a Lua script for atomicity:
EVAL "if redis.call('get', KEYS[1]) == ARGV[1] \
      then return redis.call('del', KEYS[1]) \
      else return 0 end"  1  lock:nightly-job  worker-A

# Fencing token: INCR a separate key in the SAME Lua script so acquire returns (lock, token).
```

### When Redis is not enough
- Redis master fails right after `SET NX` → replica without the key is promoted →
  two clients can each hold the lock. **Redlock** tries to mitigate by requiring a
  majority of independent nodes; still vulnerable to clock skew
  ([Kleppmann 2016](https://martin.kleppmann.com/2016/02/08/how-to-do-distributed-locking.html)).
- For correctness-critical locks (leader election, schema migration, payment
  reconciliation), use a **consensus** system (etcd, ZooKeeper) and still emit a
  fencing token.


## Real-world examples

| System                    | What they lock                                 | Backend          | Notes |
|---------------------------|------------------------------------------------|------------------|-------|
| Kubernetes controllers    | "I am the active controller for this resource" | etcd lease       | Leader election via `coordination.k8s.io/Lease`. |
| Kafka (pre-KRaft)         | "I am the controller broker"                  | ZooKeeper znode  | Ephemeral node; dies with the session. |
| Patroni (Postgres HA)     | "I am the primary"                            | etcd / Consul    | Timeouts + TTLs picked carefully. |
| Google Chubby             | Any coarse-grained lock                        | Paxos            | The paper that started the whole field. |
| Redis-backed job runners  | "Only one worker runs this cron"              | Redis `SET NX`   | Fast and good enough for most cron-like workloads. |
| HBase / Bigtable          | Region / tablet ownership                      | ZK / Chubby      | Combined with fencing via region epoch. |

### Checklist before shipping a distributed lock
1. Does the **protected resource** check a fencing token? (If not, TTLs are a lie.)
2. Is the **TTL shorter than your worst-case detection time** for a dead owner?
3. Do you have **heartbeat/renew** for long jobs?
4. Is `release` **idempotent** and **owner-checked**?
5. What happens if the **lock backend** itself fails over? (Split-brain budget.)
6. Are acquire/release paths on your **critical latency path**? (Caching, pipelining.)
